# Forecasting with RNN and LSTM



## Imports

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Dataset

In [3]:
train_text = [
    # Pattern 1: [Succeeding] + [Action] + [is] + [Fun]
    "succeeding to learn is fun",
    "succeeding to swim is fun",
    "succeeding to study is fun",
    "succeeding to play chess is fun",
    "succeeding to play music is fun",
    "succeeding to write poems is fun",
    "succeeding to win is fun",
    "succeeding to participate is fun",
    "succeeding to run is fun",
    "succeeding to code is fun",

    # Pattern 2: [Failing] + [Action] + [is] + [Bad]
    "failing to learn is bad",
    "failing to swim is bad",
    "failing to study is bad",
    "failing to play chess is bad",
    "failing to play music is bad",
    "failing to write poems is bad",
    "failing to win is bad",
    "failing to participate is bad",
    "failing to run is bad",
    "failing to code is bad",
]

test_text = [
    # Test Pattern 1: Does it associate "succeeding" with "fun"?
    "succeeding to code is",        # Expected: fun
    "succeeding to run is",         # Expected: fun
    "succeeding to play music is",  # Expected: fun

    # Test Pattern 2: Does it associate "failing" with "bad"?
    "failing to code is",           # Expected: bad
    "failing to run is",            # Expected: bad
    "failing to play music is",     # Expected: bad

    # Test Generalization:
    "succeeding to write poems is", # Expected: fun
    "failing to write poems is"     # Expected: bad
]

## Preprocessing

In [4]:
# Splits a given sentence (string) into words, and each word is translated into lower case
# E.g. given "The dog jumps", returns ["the", "dog", "jumps"]
def tokenize(sentence):
    return sentence.lower().split()

In [6]:
all_tokens = []   # empty list to contain all tokens (i.e., lower case words)
for s in train_text:
    all_tokens.extend(tokenize(s))  # for each training sentence, add to all_tokens all the lowercase words

In [7]:
# Returns a dictionary reporting, for every word, how many times it appears in the training set
counter = Counter(all_tokens)
print(counter)

Counter({'to': 20, 'is': 20, 'succeeding': 10, 'fun': 10, 'failing': 10, 'bad': 10, 'play': 4, 'learn': 2, 'swim': 2, 'study': 2, 'chess': 2, 'music': 2, 'write': 2, 'poems': 2, 'win': 2, 'participate': 2, 'run': 2, 'code': 2})


In [8]:
# Creates a vocabulary (list) containing all words in the training set, plus two special tokens
# <pad> is a token for padding: all words should have same length
# <unk> is a fallback token: if the model encounter a token not present in the training set, it uses <unk> to avoid crashing
vocab = ["<pad>", "<unk>"] + sorted(counter.keys())
print(vocab)

# Assign each word a unique integer, returns a dictionary
stoi = {w: i for i, w in enumerate(vocab)}
print(stoi)

# Assign each integer the corresponding word, returns a dictionary
itos = {i: w for w, i in stoi.items()}
print(itos)

# Compute the number of training words
vocab_size = len(vocab)

['<pad>', '<unk>', 'bad', 'chess', 'code', 'failing', 'fun', 'is', 'learn', 'music', 'participate', 'play', 'poems', 'run', 'study', 'succeeding', 'swim', 'to', 'win', 'write']
{'<pad>': 0, '<unk>': 1, 'bad': 2, 'chess': 3, 'code': 4, 'failing': 5, 'fun': 6, 'is': 7, 'learn': 8, 'music': 9, 'participate': 10, 'play': 11, 'poems': 12, 'run': 13, 'study': 14, 'succeeding': 15, 'swim': 16, 'to': 17, 'win': 18, 'write': 19}
{0: '<pad>', 1: '<unk>', 2: 'bad', 3: 'chess', 4: 'code', 5: 'failing', 6: 'fun', 7: 'is', 8: 'learn', 9: 'music', 10: 'participate', 11: 'play', 12: 'poems', 13: 'run', 14: 'study', 15: 'succeeding', 16: 'swim', 17: 'to', 18: 'win', 19: 'write'}


In [9]:
# Given a sentence, performs automatic tokenization
#
# Example
# Raw input: "Hello word"
# Tokenized: ["hello", "word"]
# stoi lookup: {"hello": 42, "world": 105, "<unk>": 0}
# Final output: [42, 105]
def encode(sentence):
    return [stoi.get(w, stoi["<unk>"]) for w in tokenize(sentence)]

In [10]:
class NextWordDataset(Dataset):

    def __init__(self, sentences, seq_len=3):
        self.samples = []
        self.seq_len = seq_len  # length of each feature sequence in the dataset (in number of words)

        # Iterate over sentences
        for s in sentences:
            tokens = encode(s)    # tokenize the sentence (tokens is a list of integers)


            # Builds a sliding window dataset
            # 3 words will represent a the features and the fourth will be the target to be predicted
            # e.g. "The cat sat on the mat" -> features: ["the", "cat", "sat"], target: ["mat"] (NOTE: words are numerically encoded, this was omitted in the
            # example for visual clarity)
            for i in range(1, len(tokens)):   # This loop defines the target's index. It starts from 1 because the first token cannot be target, as there is nothing before it
                start = max(0, i - seq_len)   # Index of the first training word
                context = tokens[start:i]     # Every word between the first training word and the target

                # The model expects sequences of same length. The first sequences will have a length smaller than seq_len
                # Example: "The cat sat on the mat" will have as first training sequence: features: ["the"], target: ["cat"]
                # The feature sequence is not long enough: we need to add padding, to make it reach seq_len
                # We add a special token for padding. The sequence will become: features: ["<pad>", "<pad>", "the"], target: ["cat"]
                if len(context) < seq_len:
                    context = [stoi["<pad>"]] * (seq_len - len(context)) + context

                # Retrieve the target token given the target index i
                target = tokens[i]

                # Save the pair (training_tokens, target_token)
                self.samples.append((context, target))

    def __len__(self):
        return len(self.samples)

    # This method returns a specific item of the dataset, given the index idx
    def __getitem__(self, idx):
        context, target = self.samples[idx]   # retrieve (training_tokens, target_token) pair

        # Convert them to PyTorch tensors and return them
        return (
            torch.tensor(context, dtype=torch.long),
            torch.tensor(target, dtype=torch.long),
        )

In [11]:
seq_len = 5   # Every feature sequence will be 5 tokens long

# Define training dataset and test dataset using the class previously defined
train_dataset = NextWordDataset(train_text, seq_len)
test_dataset = NextWordDataset(test_text, seq_len)

In [12]:
# Define training data loader and test data loader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

## Models Definition

In [13]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super().__init__()

        # 1. Built-in Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # 2. Built-in RNN block
        self.rnn = nn.RNN(
            input_size=embed_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        # 3. Built-in Linear layer
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # x shape: [batch, seq_len]

        # Transform indices to vectors: [batch, seq_len, embed_size]
        emb = self.embedding(x)

        # The RNN block handles the internal loop automatically
        # out: [batch, seq_len, hidden_size]
        # h_n: [1, batch, hidden_size] (the final hidden state)
        out, h_n = self.rnn(emb)

        # Grab the last hidden state to predict the next word
        last_hidden = out[:, -1, :]

        # Project to vocabulary scores
        logits = self.fc(last_hidden)

        return logits

In [14]:
# Define the LSTM model
class LSTMLanguageModel(nn.Module):

    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super().__init__()

        # Convert tokens into a vector of size 64 to feed them into the model
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Define LSTM block
        self.lstm = nn.LSTM(
            embed_size,         # length of embedded representation
            hidden_size,        # number of internal units to store information about the input sentence
            batch_first=True    # tells the model data comes in batches (in the form [batch_size, sequence, features])
        )

        # Fully Connected layer for final prediction
        # Outputs a probability for each word in the training set
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):

        emb = self.embedding(x)

        # Obtain the hidden state for every step in the sequence
        out, _ = self.lstm(emb)

        # Get the hidden state just for the last word
        last_hidden = out[:, -1, :]

        # Get probability for each word to be the next one
        logits = self.fc(last_hidden)

        return logits

In [15]:
# Instantiate the models
lstm_model = LSTMLanguageModel(vocab_size).to(device)
rnn_model = RNNLanguageModel(vocab_size)

## Training

In [16]:
# Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.01)
rnn_optmizer = torch.optim.Adam(rnn_model.parameters(), lr=0.01)

In [24]:
# Define the training loop
def train_epoch(model):

    # Put model in train mode (automatically enables Droput if present)
    model.train()

    total_loss = 0

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        # Forward pass
        logits = model(x)
        loss = criterion(logits, y)   # Compute loss

        # Backward pass
        optimizer.zero_grad()   # clear out old gradients
        loss.backward()         # compute how much each weight contributes to the loss
        optimizer.step()        # adjust the weights to minimize the loss

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [25]:
# Function to test the model on the test set
def evaluate(model):

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item()

    return total_loss / len(test_loader)

In [19]:
# Define number of epochs for training
epochs = 20

In [26]:
# Training the RNN
for epoch in range(epochs):

    train_loss = train_epoch(rnn_model)

    # Evaluate on test set
    test_loss = evaluate(rnn_model)

    print(f"Epoch {epoch+1}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Test loss:  {test_loss:.4f}")
    print()

Epoch 1
Train loss: 3.1328
Test loss:  3.1771

Epoch 2
Train loss: 3.1331
Test loss:  3.1771

Epoch 3
Train loss: 3.1353
Test loss:  3.1771

Epoch 4
Train loss: 3.1370
Test loss:  3.1771

Epoch 5
Train loss: 3.1316
Test loss:  3.1771

Epoch 6
Train loss: 3.1320
Test loss:  3.1771

Epoch 7
Train loss: 3.1361
Test loss:  3.1771

Epoch 8
Train loss: 3.1380
Test loss:  3.1771

Epoch 9
Train loss: 3.1395
Test loss:  3.1771

Epoch 10
Train loss: 3.1372
Test loss:  3.1771

Epoch 11
Train loss: 3.1400
Test loss:  3.1771

Epoch 12
Train loss: 3.1382
Test loss:  3.1771

Epoch 13
Train loss: 3.1345
Test loss:  3.1771

Epoch 14
Train loss: 3.1363
Test loss:  3.1771

Epoch 15
Train loss: 3.1387
Test loss:  3.1771

Epoch 16
Train loss: 3.1371
Test loss:  3.1771

Epoch 17
Train loss: 3.1393
Test loss:  3.1771

Epoch 18
Train loss: 3.1325
Test loss:  3.1771

Epoch 19
Train loss: 3.1399
Test loss:  3.1771

Epoch 20
Train loss: 3.1368
Test loss:  3.1771



In [23]:
# Training the LSTM model
for epoch in range(epochs):

    train_loss = train_epoch(lstm_model)

    # Evaluate on test set
    test_loss = evaluate(lstm_model)

    print(f"Epoch {epoch+1}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Test loss:  {test_loss:.4f}")
    print()

Epoch 1
Train loss: 1.9265
Test loss:  1.1401

Epoch 2
Train loss: 0.9462
Test loss:  0.7238

Epoch 3
Train loss: 0.7528
Test loss:  0.6628

Epoch 4
Train loss: 0.6397
Test loss:  0.7259

Epoch 5
Train loss: 0.6192
Test loss:  0.7395

Epoch 6
Train loss: 0.6043
Test loss:  0.6485

Epoch 7
Train loss: 0.6044
Test loss:  0.6797

Epoch 8
Train loss: 0.5888
Test loss:  0.6516

Epoch 9
Train loss: 0.6015
Test loss:  0.7028

Epoch 10
Train loss: 0.5773
Test loss:  0.6926

Epoch 11
Train loss: 0.5978
Test loss:  0.6965

Epoch 12
Train loss: 0.5676
Test loss:  0.6297

Epoch 13
Train loss: 0.5976
Test loss:  0.6602

Epoch 14
Train loss: 0.5677
Test loss:  0.6930

Epoch 15
Train loss: 0.6225
Test loss:  0.6672

Epoch 16
Train loss: 0.5647
Test loss:  0.6937

Epoch 17
Train loss: 0.5529
Test loss:  0.6352

Epoch 18
Train loss: 0.5929
Test loss:  0.6771

Epoch 19
Train loss: 0.5832
Test loss:  0.6316

Epoch 20
Train loss: 0.5926
Test loss:  0.6508



## Evaluation

In [27]:
# This function receives a sentence and returns next word using our trained LSTM model
def predict_next(model, text):

    # Tokenize the given sentence
    tokens = encode(text)

    # Takes the last seq_len words before the one to be predicted
    context = tokens[-seq_len:]

    # If sentence is too short, apply padding to match seq_len
    if len(context) < seq_len:
        context = [stoi["<pad>"]] * (seq_len - len(context)) + context

    # Covert the features into PyTorch tensor to be fed into the model
    x = torch.tensor(context).unsqueeze(0).to(device)


    with torch.no_grad():   # disable gradients calculation (we are not training) to speed up computations
        logits = model(x)   # get probabilities for each possible next word (i.e. all the words in the training set)

    # Extract the encoding for the word with highest probability to be the next one (according to the model)
    pred = torch.argmax(logits, dim=-1).item()

    # Given the encoding, returns the corresponding word
    return itos[pred]

In [28]:
# Testing the RNN model
print("succeeding to write poems is", predict_next(rnn_model, "succeeding to write poems is"))
print("failing to write poems is", predict_next(rnn_model, "failing to write poems is"))

print("succeeding to pass exams is", predict_next(rnn_model, "succeeding to pass exams is"))
print("failing to pass exams is", predict_next(rnn_model, "failing to pass exams is"))

succeeding to write poems is learn
failing to write poems is learn
succeeding to pass exams is to
failing to pass exams is to


In [29]:
# Testing the LSTM model
print("succeeding to write poems is", predict_next(lstm_model, "succeeding to write poems is"))
print("failing to write poems is", predict_next(lstm_model, "failing to write poems is"))

print("succeeding to pass exams is", predict_next(lstm_model, "succeeding to pass exams is"))
print("failing to pass exams is", predict_next(lstm_model, "failing to pass exams is"))

succeeding to write poems is fun
failing to write poems is bad
succeeding to pass exams is fun
failing to pass exams is bad
